# bench-imai — AI Generated Content Detection Benchmark

This notebook refreshes the CIFAKE benchmark by routing every experiment through
`BenchmarkRunner` and the shared `pipelines_torch` registries. We compare the
modern `timm` backbones introduced in `vision_models.py`, then reuse the saved
weights to audit generalisation on the DeepDetect 2025 dataset.

GPU is optional but recommended for the heavier backbones.

Note: See the final section **"8. Analysis and Lessons Learned"** for a
concise summary of results, cross-dataset generalisation findings and recommended
next steps.


## 1. Environment setup
Install optional vision dependencies (`timm`, `kaggle`) so the registry backbones
load without manual package management.


In [ ]:
import os, subprocess, sys
from pathlib import Path

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "ml_pipeline" else cwd
if (repo_root / "ml_pipeline").is_dir() and (repo_root / "requirements.txt").is_file():
    os.chdir(repo_root)
    print(f"Using local checkout: {repo_root}")
else:
    subprocess.run(["rm", "-rf", "bench_research_ml_project"], check=True)
    subprocess.run(["git", "clone", "https://github.com/oremaz/bench_research_ml_project"], check=True)
    os.chdir("bench_research_ml_project")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

    subprocess.run([sys.executable, "-m", "pip", "uninstall", "numpy", "scipy", "scikit-learn", "-y"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==1.24.3", "scipy==1.10.1", "scikit-learn==1.3.0"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.9", "kaggle", "rich"], check=True)

In [ ]:
from pathlib import Path
import os

if Path.cwd().name != "ml_pipeline":
    os.chdir("ml_pipeline")
print(f"Working directory: {Path.cwd()}")

## 2. Imports and deterministic utilities
Everything important (models, metrics, benchmarking) is imported from the shared
library so we avoid redefining models or augmentations inside the notebook.


In [ ]:
import os
from pathlib import Path
from typing import Iterable
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.vision_models import MODEL_REGISTRY as VISION_MODELS
from pipelines_torch.base import SimplePredictor
from utils.metrics import METRIC_REGISTRY
from utils.utils import load_model, load_model_by_name, RESULTS_DIR
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 3. Download and prepare CIFAKE
The helper below pulls CIFAKE from Kaggle and converts the PyTorch `ImageFolder`
structure into NumPy arrays so that `BenchmarkRunner` can operate on it.


In [ ]:
from pathlib import Path

from utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_DATASET = "birdy654/cifake-real-and-ai-generated-synthetic-images"
DATA_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_DATASET,
    local_dir=Path("data/cifake"),
    description="CIFAKE dataset",
    kaggle_subdir="cifake-real-and-ai-generated-synthetic-images",
)

if (DATA_DIR / "train").exists():
    print(f"✅ CIFAKE data available at {DATA_DIR}")
else:
    print(f"⚠️ CIFAKE dataset missing expected 'train' directory at {DATA_DIR}")

In [ ]:
IMG_SIZE = 32
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dir = DATA_DIR / "train"
val_dir = DATA_DIR / "test"
if not train_dir.exists() or not val_dir.exists():
    raise FileNotFoundError("Expected CIFAKE to expose train/ and test/ splits")

train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

class_names = train_ds.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")

In [ ]:
def dataset_to_numpy(dataset: datasets.ImageFolder):
    tensors = [img for img, _ in dataset]
    X = torch.stack(tensors).numpy()
    y = np.array(dataset.targets, dtype=np.int64)
    return X.astype(np.float32), y

X_all, y_all = dataset_to_numpy(train_ds)
X_test, y_test = dataset_to_numpy(val_ds)

In [ ]:
print(f"Test shape: {X_test.shape}")

## 4. Configure `BenchmarkRunner`
We select the modern `timm` backbones registered in `vision_models.py` and
evaluate macro metrics from `utils.metrics`.


In [ ]:
models = []
for model in VISION_MODELS:
    if model not in ["qwen2_vl_qlora"]:
        models.append(model)

model_configs = []
epochs = {}
for name in models:
    if name not in VISION_MODELS:
        raise KeyError(f"{name} is not registered in pipelines_torch.vision_models")
    model_configs.append({
        "name": name,
        "class": VISION_MODELS[name],
        "params": {"num_classes": num_classes},
    })
    epochs[name] = 7


runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    task_type="classification",
    device=DEVICE,
    epochs=epochs,
    batch_size=32,
    early_stopping=None,
    use_class_weights=True,
    use_kfold=False,
    learning_rate=3e-4,
    path_start="bench_imai",
    random_state=SEED,
)
results_df = runner.run(X_all, y_all)
results_df

In [ ]:
import os
import tarfile
from IPython.display import FileLink

# Change to the results directory
os.chdir('/kaggle/working/results')

# Create tar.gz archive containing all .pt files
with tarfile.open('/kaggle/working/all_files.tar.gz', 'w:gz') as tar:
    for root, dirs, files in os.walk('.'):
        for file in files:
            tar.add(os.path.join(root, file))

# Generate download link
os.chdir('/kaggle/working')
FileLink('all_files.tar.gz')


## 6. Evaluate the CIFAKE test split
Re-use the same helper to score the official CIFAKE `test/` directory.


In [ ]:
def evaluate_saved_models(model_names: Iterable[str], X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
        "roc_auc": "roc_auc",
        "pr_auc": "pr_auc",
    }
    records = []
    for name in model_names:
        try:
            model = load_model_by_name(VISION_MODELS[name], name, {"num_classes": num_classes}, path_start="bench-imai-final/bench_imai")
        except FileNotFoundError:
            print(f"⚠️ Skipping {name}: checkpoint not found")
            continue
        print(name)
        predictor = SimplePredictor(model, task_type="classification", device=DEVICE, batch_size=32)
        probs = predictor.predict_proba(X)
        scores = {
            label: float(METRIC_REGISTRY[key](y, probs))
            for label, key in metric_map.items()
        }
        records.append({
            "model": name,
            **scores,
        })
    return pd.DataFrame.from_records(records)

In [ ]:
models = []
for model in VISION_MODELS:
    if model not in ["qwen2_vl_qlora"]:
        models.append(model)
cifake_test_metrics = evaluate_saved_models(models, X_test, y_test)
cifake_test_metrics.sort_values("accuracy", ascending=False)

## 7. shoes-dataset 2025 generalisation check
The repository previously evaluated CIFAKE models on the shoes-dataset.
We keep that workflow: download the dataset from Kaggle (if necessary), locate an
`ImageFolder`-compatible split, and score it with the same predictor helper.

In [ ]:
from pathlib import Path
from utils.kaggle_utils import ensure_kaggle_dataset

# Kaggle dataset: https://www.kaggle.com/datasets/sunnykakar/shoes-dataset-real-and-ai-generated-images
KAGGLE_SHOES = "sunnykakar/shoes-dataset-real-and-ai-generated-images"

# ✅ Use the actual extracted top-level folder name from the dataset zip:
EXPECTED_TOPDIR = "shoes-dataset-real-and-ai-generated-images"

SHOES_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_SHOES,
    local_dir=Path("data/shoes-ai-vs-real"),
    description="SunnyKakar Shoes (AI vs Real)",
    kaggle_subdir=EXPECTED_TOPDIR,   # <— was "shoes-dataset" before; that's wrong for this dataset
)

if SHOES_DIR.exists():
    print(f"✅ Shoes dataset available at {SHOES_DIR}")
else:
    print(f"⚠️ Shoes dataset missing at {SHOES_DIR}")

def is_imagefolder_dir(path: Path) -> bool:
    path = Path(path)
    if not path.is_dir():
        return False
    subdirs = [p for p in path.iterdir() if p.is_dir()]
    if len(subdirs) < 2:
        return False
    # At least one class subdir must contain files
    return any(any(child.is_file() for child in d.iterdir()) for d in subdirs)

def find_imagefolder_split(base_dir: Path) -> Path:
    base_dir = Path(base_dir)

    # If the provided base is already an ImageFolder root, use it.
    if is_imagefolder_dir(base_dir):
        return base_dir

    # Common split folder names
    preferred = ("train", "test", "validation", "val", "eval", "holdout")
    for name in preferred:
        for variant in {name, name.upper(), name.capitalize()}:
            candidate = base_dir / variant
            if is_imagefolder_dir(candidate):
                return candidate

    # Extra: If ensure_kaggle_dataset returned local_dir but the data is nested one level deeper,
    # look for a folder that contains both 'ai-midjourney' and 'real' with images inside.
    for candidate in base_dir.glob("*"):
        if candidate.is_dir():
            ai_dir = candidate / "ai-midjourney"
            real_dir = candidate / "real"
            if ai_dir.exists() and real_dir.exists() and is_imagefolder_dir(candidate):
                return candidate

    # General recursive search as a final fallback.
    for candidate in sorted(base_dir.rglob("*")):
        if is_imagefolder_dir(candidate):
            return candidate

    raise ValueError(f"Could not locate an ImageFolder split inside {base_dir}")

IMG_SIZE = 32
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

models = []
for model in VISION_MODELS:
    if model not in ["qwen2_vl_qlora"]:
        models.append(model)

def evaluate_imagefolder_split(split_dir: Path, dataset_name: str) -> pd.DataFrame:
    dataset = datasets.ImageFolder(split_dir, transform=transform)
    if len(dataset) == 0:
        raise ValueError(f"No samples detected under {split_dir}")
    X_eval, y_eval = dataset_to_numpy(dataset)
    df = evaluate_saved_models(models, X_eval, y_eval)
    return df

shoes_split = find_imagefolder_split(SHOES_DIR)
print(f"Using {shoes_split} for evaluation")
shoes_metrics = evaluate_imagefolder_split(shoes_split, "Shoes: Real vs AI (SunnyKakar)")
shoes_metrics.sort_values("accuracy", ascending=False)

# 8. Analysis and Lessons Learned

## Overview of Results

This benchmark evaluated **10 vision models** on AI-generated vs. Real image detection across two datasets:
- **CIFAKE (Training)**: 60,000 train + 20,000 test images (CIFAR-10 style, AI-generated via StyleGAN)
- **Shoes Dataset (Generalization Test)**: Real vs. AI-generated shoe images

### Model Performance on CIFAKE Test Set

| Rank | Model | Accuracy | F1-Macro | ROC-AUC | Architecture Type |
|------|-------|----------|----------|---------|-------------------|
| 1 | **efficientnet** | **97.1%** | 97.1% | 99.6% | EfficientNet-B4 + specialized fake detection |
| 2 | adaptive_cnn | 96.2% | 96.2% | 99.3% | Custom adaptive CNN |
| 3 | timm_convnextv2_tiny | 96.0% | 96.0% | 99.3% | ConvNeXt-V2 (modern ConvNet) |
| 4 | residual_cnn | 95.8% | 95.8% | 99.2% | Custom residual CNN |
| 5 | clip_classifier | 94.6% | 94.6% | 98.8% | CLIP ViT-B/32 (multimodal) |
| 10 | timm_vit_base_patch16 | 87.9% | 87.9% | 95.5% | Standard ViT-Base |

**Key Findings:**
- **EfficientNet dominates** with 97.1% accuracy using the EfficientNet-B4 backbone
- **Custom CNNs outperform** standard pretrained models (adaptive_cnn: 96.2% vs resnet50: 90.2%)
- **Vision Transformers struggle** on this task (ViT-Base: 87.9%, ViT-MAE: 90.4%) - likely due to CIFAKE's low resolution (32x32)

---

## Cross-Dataset Generalization Analysis

### Performance Drop on Shoes Dataset

| Model | CIFAKE Accuracy | Shoes Accuracy | **Drop** | Generalization Score |
|-------|----------------|----------------|----------|---------------------|
| clip_classifier | 94.6% | **79.1%** | **-15.5%** | ⭐⭐⭐ Best |
| resnet50 | 90.2% | 71.9% | -18.3% | ⭐⭐ Good |
| timm_vit_base_patch16 | 87.9% | 64.0% | -23.9% | ❌ Poor |
| efficientnet | **97.1%** | 60.1% | **-37.0%** | ❌ Poor |
| adaptive_cnn | 96.2% | 55.2% | **-40.9%** | ❌ Severe Overfitting |

**Critical Insights:**
1. **CLIP generalizes best** (79.1% on shoes) - multimodal pretraining on 400M diverse images provides robustness
2. **Specialized models collapse** - EfficientNet (-37%) and custom CNNs (-40%+) overfit to CIFAKE's low-res (32x32) StyleGAN artifacts
3. **Distribution shift is massive** - resolution mismatch (32x32 → high-res) and multi-generator diversity cause 15-40% performance drops

---

## Key Takeaways

### ✅ Successes
1. **High in-domain accuracy** - Multiple models achieve >95% on CIFAKE test set
2. **Multimodal models generalize** - CLIP demonstrates superior cross-domain robustness

### ❌ Limitations & Challenges
1. **Single-generator training insufficient** - Models learn StyleGAN-specific patterns, fail on diverse generators (15-40% drop)
2. **Lack of robustness evaluation** - No adversarial testing, compression artifacts, or post-processing variations tested
3. **In-domain accuracy ≠ Generalization** - High CIFAKE scores (96%+) don't predict cross-dataset performance

---

## Next Steps & Recommendations

### 1. **Training on Artifact Dataset** 🎯
- **Dataset**: [Artifact Dataset (Kaggle)](https://www.kaggle.com/datasets/awsaf49/artifact-dataset)
- **Why**: Includes images from **multiple generative models** (Stable Diffusion, Midjourney, DALL-E, etc.)
- **Expected Impact**: Train on diverse artifacts → reduce overfitting to single generator

### 2. **Resolution-Adaptive Training**
- Test models with varying input resolutions (32x32 → 224x224 → 512x512)
- Use progressive training: start with low-res CIFAKE, fine-tune on high-res Artifact data
- Evaluate whether resolution-invariant features improve generalization

### 3. **Ensemble & Hybrid Approaches**
- **Ensemble**: Combine CLIP (generalization) + EfficientNet (in-domain accuracy)
- **Hybrid**: CLIP features + custom CNN classifier head
- **Stacking**: Use CIFAKE predictions as features for meta-learner on Shoes dataset

### 4. **Frequency-Domain Analysis**
- Implement FFT-based fake detection (AI images often have frequency anomalies)
- Compare spectral features vs. learned CNN features
- Explore hybrid spatial + frequency models (e.g., F3Net, SRM)

### 5. **Interpretability & Explainability**
- Grad-CAM visualizations: Where do models look for fake cues?
- Attention map analysis for ViTs: Do they focus on compression artifacts vs. semantic inconsistencies?
- Feature space analysis: t-SNE/UMAP of real vs. fake embeddings across datasets

### 6. **Benchmark Expansion**
- Add more datasets: **GenImage**, **DIRE** (diffusion-generated images), **FaceForensics++**
- Include video-based fake detection (Deepfake videos from FaceForensics++)
- Multi-domain evaluation: faces, objects, scenes, text-to-image outputs

### 7. **Model Architecture Exploration**
- **EfficientNet-V2-L** - test larger variants of EfficientNet family
- **DINOv2** - self-supervised ViT with strong generalization

---

## Conclusion

This benchmark reveals a **critical gap** between in-domain accuracy and real-world generalization in AI-generated image detection. While specialized models achieve near-perfect performance on CIFAKE (97%), they catastrophically fail on out-of-distribution data (60% on Shoes). 

**The path forward** requires:
1. Training on **multi-generator datasets** (Artifact)
2. Prioritizing **robust pretrained models** (CLIP) over custom architectures
3. Systematic **cross-dataset evaluation** as a standard practice

The next iteration should focus on the **Artifact dataset** to address the single-generator overfitting problem and establish models that generalize across diverse AI generation techniques.